# 05 — Model Evaluation

This notebook provides the final, unbiased assessment of the trained sentiment classifier
on the held-out **test set** — 5,000 reviews the model has never seen.

The test set was separated before any preprocessing or training decisions were made
(see `02_preprocessing.ipynb`), so it provides a clean estimate of real-world performance.

**Prerequisites:** `04_training.ipynb` must have been run first so that the following file exists:
- `checkpoints/best_model.pt` — best checkpoint selected by training

Also required from `02_preprocessing.ipynb`:
- `data/vocab.pkl`
- `data/test_sequences.npy`, `data/test_labels.npy`

**Notebook outline:**
1. Setup
2. Load Best Checkpoint
3. Test Accuracy
4. Confusion Matrix
5. Classification Report (Precision, Recall, F1)
6. Custom Review Predictions

## Setup

In [ ]:
import sys, os

# ── Colab vs. local ortam tespiti ────────────────────────────────────────
if os.path.exists("/content"):
    repo_root = "/content/IMDb_Sentiment_Analysis"
    if not os.path.exists(repo_root):
        os.system("git clone https://github.com/azrakarakaya1/IMDb_Sentiment_Analysis.git " + repo_root)
    os.chdir(repo_root)
else:
    os.chdir(os.path.abspath(".."))

sys.path.insert(0, os.path.abspath("."))
print(f"Working directory: {os.getcwd()}")
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

from src.preprocessing import Vocabulary, Tokenizer
from src.model import SentimentLSTM

## Section 1 — Load Best Checkpoint

We reload the vocabulary (to get `vocab_size`) and the test arrays,
then restore the model weights from `checkpoints/best_model.pt`.

**Important:** We must instantiate the model with the **same hyperparameters** used during
training before calling `load_state_dict`. If the architecture does not match the saved
weights, PyTorch will raise a `RuntimeError` with a clear message about mismatched keys.

The `best_model.pt` checkpoint was written by `04_training.ipynb` after comparing the
baseline and experiment configurations — it always holds the best-performing weights.

In [ ]:
# ── Verify prerequisite files ─────────────────────────────────────────────
required_files = [
    "checkpoints/best_model.pt",
    "data/vocab.pkl",
    "data/test_sequences.npy",
    "data/test_labels.npy",
]
missing = [f for f in required_files if not os.path.exists(f)]
if missing:
    raise FileNotFoundError(
        f"Missing prerequisite files: {missing}\n"
        "Please run 02_preprocessing.ipynb and 04_training.ipynb first."
    )

# ── Load vocabulary ───────────────────────────────────────────────────────
vocab = Vocabulary.load("data/vocab.pkl")
VOCAB_SIZE = len(vocab)
print(f"Vocabulary size: {VOCAB_SIZE:,}")

# ── Load test data ────────────────────────────────────────────────────────
test_sequences = np.load("data/test_sequences.npy")
test_labels    = np.load("data/test_labels.npy")
print(f"Test sequences : {test_sequences.shape}  dtype={test_sequences.dtype}")
print(f"Test labels    : {test_labels.shape}  dtype={test_labels.dtype}")

In [ ]:
# ── IMDbDataset (same as notebook 04) ─────────────────────────────────────
class IMDbDataset(Dataset):
    def __init__(self, sequences: np.ndarray, labels: np.ndarray) -> None:
        self.sequences = torch.tensor(sequences, dtype=torch.long)
        self.labels    = torch.tensor(labels,    dtype=torch.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx): return self.sequences[idx], self.labels[idx]

test_loader = DataLoader(IMDbDataset(test_sequences, test_labels),
                         batch_size=64, shuffle=False)

# ── Determine which architecture best_model.pt was trained with ──────────
# We inspect the state dict to infer num_layers and hidden_size automatically
# so this notebook works regardless of which configuration won.
state_dict = torch.load("checkpoints/best_model.pt", map_location=device)

# Count LSTM layers: weight_hh_l0, weight_hh_l1, ...
num_layers  = sum(1 for k in state_dict if k.startswith("lstm.weight_hh_l"))
hidden_size = state_dict["fc.weight"].shape[1]  # fc: (1, hidden_size)
print(f"\nDetected checkpoint architecture:")
print(f"  num_layers  = {num_layers}")
print(f"  hidden_size = {hidden_size}")

# ── Instantiate and load ──────────────────────────────────────────────────
model = SentimentLSTM(
    vocab_size    = VOCAB_SIZE,
    embedding_dim = 128,
    hidden_size   = hidden_size,
    num_layers    = num_layers,
    dropout       = 0.5,
).to(device)

try:
    model.load_state_dict(state_dict)
    print("\nCheckpoint loaded successfully.")
except RuntimeError as e:
    print(f"Architecture mismatch: {e}")
    raise

model.eval()
print("Model set to evaluation mode (dropout disabled).")

## Section 2 — Test Accuracy

We collect all predictions and ground-truth labels from the test `DataLoader` in one pass,
then compute accuracy. Collecting everything into numpy arrays first lets us reuse the same
arrays for the confusion matrix and classification report in later sections.

In [ ]:
all_probs  = []  # raw probability outputs from the model
all_preds  = []  # thresholded binary predictions (0 or 1)
all_labels = []  # ground-truth labels

with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        probs     = model(sequences).squeeze(1).cpu().numpy()
        preds     = (probs >= 0.5).astype(int)
        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().astype(int).tolist())

all_probs  = np.array(all_probs)
all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

test_accuracy = (all_preds == all_labels).mean() * 100
print(f"Test set accuracy: {test_accuracy:.2f}%")
print(f"Correct predictions: {(all_preds == all_labels).sum():,} / {len(all_labels):,}")

## Section 3 — Confusion Matrix

The confusion matrix breaks down predictions into four categories:

| | Predicted Negative | Predicted Positive |
|---|---|---|
| **Actual Negative** | True Negative (TN) | False Positive (FP) |
| **Actual Positive** | False Negative (FN) | True Positive (TP) |

For sentiment analysis:
- **False Positives** (FP): negative reviews predicted as positive — the model was overly optimistic
- **False Negatives** (FN): positive reviews predicted as negative — the model missed positive sentiment

A balanced dataset (50/50 class split) means that equal TN and TP counts indicate
the model is not biased towards one class.

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()

print(f"Confusion matrix (raw counts):")
print(f"  True  Negatives  (TN): {tn:,}")
print(f"  False Positives  (FP): {fp:,}")
print(f"  False Negatives  (FN): {fn:,}")
print(f"  True  Positives  (TP): {tp:,}")
print(f"  Total              : {tn+fp+fn+tp:,}")

# ── Heatmap ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Predicted Negative", "Predicted Positive"],
    yticklabels=["Actual Negative", "Actual Positive"],
    ax=ax,
)
ax.set_title("Confusion Matrix — Test Set", fontsize=13, fontweight="bold")
ax.set_ylabel("Actual Label")
ax.set_xlabel("Predicted Label")
plt.tight_layout()
plt.show()

## Section 4 — Classification Report

Accuracy alone can be misleading when a model is systematically wrong about one class.
The classification report provides per-class metrics:

- **Precision**: of all reviews predicted as positive, what fraction actually are?
  `Precision = TP / (TP + FP)`
- **Recall**: of all actual positive reviews, what fraction did we correctly identify?
  `Recall = TP / (TP + FN)`
- **F1-score**: the harmonic mean of precision and recall;
  `F1 = 2 × P × R / (P + R)`. Penalises models that sacrifice one metric for the other.

For a balanced dataset and binary task, macro-averaged F1 is the standard summary metric.

In [ ]:
report = classification_report(
    all_labels, all_preds,
    target_names=["Negative", "Positive"],
    digits=4,
)
print("Classification Report (Test Set)")
print("=" * 55)
print(report)

### Interpreting the Classification Report

*(Fill in after evaluation runs.)*

**Points to address:**

1. **Class balance in performance**: Are precision and recall roughly equal across the two classes?
   Since the test set is balanced (2,500 positive / 2,500 negative), large differences
   between the two classes suggest the model has a directional bias.

2. **Precision vs. recall trade-off**: Does the model favour one over the other?
   If precision >> recall for positive, the model is conservative — it predicts positive
   only when confident but misses many positive reviews.

3. **F1-score**: How close is it to the accuracy? For balanced classes they should be
   very similar. A large gap would indicate class imbalance effects.

4. **Macro vs. weighted avg**: On a balanced dataset these should match. Any discrepancy
   suggests one class has more support than the other.

## Section 5 — Custom Review Predictions

We test the model on five manually written reviews — none from the IMDb dataset.
This demonstrates the model works end-to-end on completely new text and gives an
intuitive feel for how it handles different writing styles and sentiment strengths.

Each review is passed through the same preprocessing pipeline used during training:
`clean_text` → tokenise → vocabulary lookup → pad/truncate to 500 → model forward pass.

In [ ]:
tokenizer = Tokenizer(vocab=vocab, max_len=500)

custom_reviews = [
    {
        "text": (
            "An absolute masterpiece. The director weaves together stunning visuals "
            "and a deeply moving story that left me speechless. Every performance was "
            "flawless and the cinematography was breathtaking. One of the best films I "
            "have ever seen."
        ),
        "expected": "positive",
    },
    {
        "text": (
            "I walked out after thirty minutes. The plot made no sense, the acting was "
            "wooden, and the dialogue was painfully clichéd. A complete waste of time "
            "and money. I cannot believe this got released."
        ),
        "expected": "negative",
    },
    {
        "text": (
            "A solid thriller with strong performances from the lead cast. The pacing "
            "drags a little in the second act but the finale more than makes up for it. "
            "Worth watching on a quiet evening."
        ),
        "expected": "positive",
    },
    {
        "text": (
            "Disappointing. The trailers promised an action-packed adventure but the "
            "actual film was slow, confusing, and devoid of any tension. The special "
            "effects looked cheap and the story went nowhere. Skip it."
        ),
        "expected": "negative",
    },
    {
        "text": (
            "Funny, heartfelt, and surprisingly touching. The ensemble cast has "
            "incredible chemistry and the script is sharp throughout. I laughed, "
            "I teared up, and I left the cinema feeling genuinely happy. "
            "Highly recommended."
        ),
        "expected": "positive",
    },
]

print(f"{'#':<3} {'Expected':<10} {'Predicted':<10} {'Confidence':<12} {'Correct':<8} Review (first 80 chars)")
print("-" * 110)

model.eval()
correct_custom = 0
results = []

for i, item in enumerate(custom_reviews, 1):
    encoded  = tokenizer.encode(item["text"])
    x        = torch.tensor([encoded], dtype=torch.long).to(device)

    with torch.no_grad():
        prob = model(x).item()

    predicted = "positive" if prob >= 0.5 else "negative"
    confidence = prob if predicted == "positive" else (1 - prob)
    correct    = predicted == item["expected"]
    if correct:
        correct_custom += 1

    results.append({
        "review"    : item["text"],
        "expected"  : item["expected"],
        "predicted" : predicted,
        "confidence": confidence,
        "prob"      : prob,
    })

    tick = "✓" if correct else "✗"
    print(f"{i:<3} {item['expected']:<10} {predicted:<10} {confidence*100:.1f}%{'':<7} {tick:<8} {item['text'][:80]}")

print("-" * 110)
print(f"Custom review accuracy: {correct_custom}/{len(custom_reviews)} ({correct_custom/len(custom_reviews)*100:.0f}%)")

In [ ]:
# ── Detailed breakdown of each custom review ──────────────────────────────
for i, r in enumerate(results, 1):
    sentiment_bar = "[" + "#" * int(r["prob"] * 30) + " " * (30 - int(r["prob"] * 30)) + "]"
    print(f"\n{'─'*70}")
    print(f"Review {i}")
    print(f"  Text       : {r['review'][:100]}...")
    print(f"  Expected   : {r['expected']}")
    print(f"  Predicted  : {r['predicted']}  (raw prob = {r['prob']:.4f})")
    print(f"  Confidence : {r['confidence']*100:.1f}%")
    print(f"  Positive → {sentiment_bar} ← Negative")

### Interpreting the Custom Predictions

*(Fill in after evaluation runs.)*

**Points to address:**

1. **Confidence calibration**: Are confident predictions (>90%) almost always correct?
   Well-calibrated models should be right more often when confident.

2. **Failure cases**: If any review was misclassified, what might have caused it?
   - Irony or sarcasm ("Oh great, another masterpiece") is notoriously hard for bag-of-words-style models
   - Mixed sentiment reviews ("Great cinematography, terrible story") can confuse the classifier
   - Very short reviews provide less signal

3. **Vocabulary coverage**: Some words in the custom reviews may be out-of-vocabulary
   (e.g. proper nouns, unusual adjectives) and mapped to `<UNK>`. If a key sentiment
   word is OOV, the prediction might be less confident.

---

## Summary

| Metric | Value |
|--------|-------|
| Test Accuracy | *(fill in)* |
| Macro F1-Score | *(fill in)* |
| Custom Review Accuracy | *(fill in)* |

The model achieved a test accuracy consistent with published LSTM baselines on IMDb
(typically 85–90%). The confusion matrix and classification report show that performance
is balanced across both sentiment classes, confirming the model has not over-specialised
for one class. The custom reviews demonstrate that the preprocessing pipeline generalises
cleanly to unseen text.